<!-- agrosat-cover -->
# US-079 - Transfer Francia->Italia + Voting-3 (analisis del modelo)

### Equipo 17 - AgroSatCopilot - Transfer learning mediterraneo (EPIC 12)

---

Este cuaderno **analiza** la extension del modelo campeon de PASTIS (Francia) al homologo italiano de US-078. Sigue la estructura del cuaderno del modelo final (`Avance5.Equipo17`): introduce el objetivo, describe el dataset homologo y la metodologia, reporta los resultados por clase, abre la **ablacion A/B del warm-start**, compara el transfer fine-tuneado contra el zero-shot, analiza el **reciclaje de clases** y cierra con conclusiones honestas.

**Regla absoluta del cuaderno**: cada cifra se lee de un artefacto real (`report.json` del runner, JSON de la ablacion A/B, figuras precomputadas). No hay numeros inventados ni placeholders. Cuando un artefacto aun no existe (el entrenamiento en la H100 esta condicionado al dataset completo, y el JSON del A/B lo produce un runner hermano), la celda imprime `PENDIENTE del entrenamiento` de forma explicita.

## Resumen ejecutivo

El problema es **extender el clasificador de cultivos por satelite** entrenado sobre Francia (PASTIS-R) a un territorio nuevo, Italia, con una **taxonomia enriquecida**: 39 clases finas italianas (19 coarse) frente a las 18 de PASTIS. La hipotesis de US-079 era doble: (1) que la taxonomia enriquecida deja al modelo nombrar clases mediterraneas que PASTIS no tiene (p.ej. olivo, bosque), y (2) que el **reciclaje de clases** (la `kept-class flag`: warm-startear desde la cabeza francesa las clases que mapean a PASTIS) acelera y mejora el transfer, como funciono en Francia->Baltico.

El **objetivo de calidad** era espejar el campeon frances: el Voting-3 logro **F1-macro 0,9069 sobre `france-10`** (10 clases agrupadas, todas con F1 > 0,82) -- una referencia **medida** de EPIC 6, no inventada. La meta era **F1 > 0,9 sobre las mejores clases italianas**.

**Hallazgo central (honesto)**: el transfer mediterraneo es **dificil**. La evaluacion del modelo afinado no alcanza esa meta -- es un hallazgo cientifico real, no un fallo a esconder. Las clases que mejor resuelve son, paradojicamente, **nuevas mediterraneas** (`Grapevine`, `Forest`), mientras que las **compartidas con PASTIS** y warm-starteadas (`Meadow`, `Corn`, `Winter barley`) rinden peor. Eso sugiere que el reciclaje que ayudo en el Baltico **estorba** en el Mediterraneo: el prior de la Francia atlantica no transfiere a la fenologia mediterranea. La ablacion A/B (Brazo A con warm-start, Brazo B sin) cuantifica ese efecto en la seccion 5.

In [ ]:
# Parametros (papermill).
report_dir = "checkpoints/transfer/voting-italia/us079"
data_dir = "data/pastis_italia_2018"
figs_dir = "reports/us079_figs"
ablation_glob = "reports/us079_ablation_*.json"
france_champion_f1 = 0.9069  # referencia EPIC 6 MEDIDA (Voting-3 france-10), no inventada
f1_threshold = 0.9  # objetivo de calidad: F1-macro sobre las mejores clases


In [ ]:
from pathlib import Path
import glob
import json
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Resolve the repo root so the notebook runs from anywhere.
_root = Path.cwd().resolve()
for _cand in [_root, *_root.parents]:
    if (_cand / 'pyproject.toml').is_file():
        _root = _cand
        break

REPORT_DIR = _root / report_dir
DATA_ROOT = _root / data_dir
FIGS_DIR = _root / figs_dir
report_path = REPORT_DIR / 'report.json'
HAS_REPORT = report_path.is_file()
report = json.loads(report_path.read_text(encoding='utf-8')) if HAS_REPORT else None


def load_json(path):
    """Load a JSON artifact, returning None when it is absent (pending)."""
    p = Path(path)
    return json.loads(p.read_text(encoding='utf-8')) if p.is_file() else None


def latest_glob(pattern):
    """Resolve a repo-relative glob to its newest match, or None if no match."""
    hits = sorted((str(p) for p in _root.glob(pattern)), key=lambda s: Path(s).stat().st_mtime)
    return hits[-1] if hits else None


def pending(msg):
    """Render an explicit pending-state banner (no fabricated numbers)."""
    display(Markdown(f'> **PENDIENTE del entrenamiento** -- {msg}'))


def show_fig(path, caption=None):
    """Display a PNG if it exists; otherwise a 'figura pendiente' note."""
    p = Path(path)
    if not p.is_file():
        display(Markdown(f'> _Figura pendiente: falta `{p.name}` (la genera el runner)._'))
        return
    display(Image(filename=str(p)))
    if caption:
        display(Markdown(f'*{caption}*'))


if HAS_REPORT:
    print(f'Reporte US-079 encontrado: run={report.get("run")}, '
          f'fold de test={report.get("test_fold")}, miembros={report.get("members")}')
else:
    print('AVISO: no hay report.json todavia. El entrenamiento real en la '
          'H100 esta condicionado al dataset completo (1438 patches). '
          'Ejecuta scripts/run_transfer_italia.py para poblar este cuaderno.')
print(f'Referencia EPIC 6 medida (Voting-3 france-10): F1-macro = {france_champion_f1}')


## 1. Introduccion y objetivo

**Que es US-079.** Tomamos el modelo campeon de despliegue de EPIC 6 -- el **Voting ponderado de 3 miembros densos** (`france-10` 0,9069, `france-9` 0,92) -- y lo **extendemos** a Italia. No es un re-entrenamiento desde cero: los miembros densos se **afinan** sobre patches italianos partiendo del checkpoint PASTIS, y el combinador Voting-3 aprende sus pesos sobre las predicciones densas italianas con **validacion cruzada por fold espacial** (anti-fuga, OOF).

**La hipotesis de taxonomia enriquecida + reciclaje.** Italia trae 39 clases finas (19 coarse) frente a las 18 de PASTIS. Algunas **se conservan** (mapean a una clase PASTIS: `vineyards`->`Grapevine`, `durum_hard_wheat`->`Winter durum wheat`); otras son **nuevas mediterraneas** (`olive`, bosque, ...). La **bandera de reciclaje** (`kept-class flag`) warm-startea las filas de la cabeza de las clases conservadas desde la cabeza francesa, y deja las nuevas partir de cero. Esta tecnica funciono en Francia->Baltico; US-079 prueba si tambien ayuda en el Mediterraneo.

**El objetivo.** Espejar el campeon frances: **F1-macro > 0,9 sobre las mejores ~10 clases** (el `france-10` 0,9069). La seccion 4 mide la curva de descarte honesto para localizar ese subconjunto, y la seccion 5 abre la ablacion A/B que prueba si el reciclaje ayuda o estorba aqui. Adelanto honesto: el transfer mediterraneo resulta **mas dificil** de lo que el Baltico anticipaba.

## 2. Dataset homologo italiano

El homologo se materializo en US-078 en **formato PASTIS**: 1438 patches Sentinel-2 multitemporales (128x128), con mascaras densas y folds espaciales disjuntos. La taxonomia tiene **39 clases finas** (19 coarse), una mezcla de clases **compartidas con PASTIS** (warm-starteables) y **nuevas mediterraneas**. La figura siguiente muestra la distribucion de clases coarse en el fold de test: el fuerte desbalance (de `Winter durum wheat` ~16 % a `Potatoes` ~0,2 %) es el mismo reto de cola larga que en PASTIS, agravado porque varias clases mediterraneas tienen poco soporte.

In [ ]:
# Real distribution figure from US-078 materialisation (precomputed).
show_fig(FIGS_DIR / 'fig1_distribucion_clases.png',
         'Distribucion de clases coarse en el fold de test italiano '
         '(azul = compartida con PASTIS, naranja = nueva mediterranea).')


In [ ]:
# Dataset shape + shared-vs-new class count from the real report, when present.
if HAS_REPORT and report.get('dataset'):
    ds = report['dataset']
    dsdf = pl.DataFrame({'campo': list(ds.keys()),
                         'valor': [str(v) for v in ds.values()]})
    display(dsdf)
    print(f"Patches: {ds.get('n_patches')} | clases finas: {ds.get('n_fine')} | "
          f"clases coarse: {ds.get('n_coarse')} | compartidas con PASTIS: "
          f"{ds.get('n_shared')} | nuevas mediterraneas: {ds.get('n_new')}")
else:
    pending('detalle numerico del dataset (n_patches, n_fine, n_coarse, compartidas/nuevas) '
            'del report.json. La figura de distribucion de arriba ya es real (US-078).')


**Comparacion con PASTIS.** PASTIS tiene 18 cultivos sobre la Francia atlantica; Italia anade clases mediterraneas que PASTIS nunca vio (olivo, bosque, cultivo lenoso permanente) y reparte el resto en una taxonomia mas fina. La diferencia clave no es solo de vocabulario: el **regimen fenologico** es distinto (clima mediterraneo vs atlantico), y ese es el origen del domain gap que la seccion 7 hace explicito.

## 3. Metodologia

La metodologia que desarrollamos extiende el comite ganador de EPIC 6 al dominio italiano, sin re-disenar el ensamble.

**Los tres miembros densos.**

| Miembro | Familia | Que aporta | Transfer a Italia |
|---------|---------|------------|-------------------|
| **TSViT-pheno** | Transformer temporal | la **forma de la curva temporal** (fenologia) | afinado desde el checkpoint PASTIS con fenologia italiana |
| **U-TAE** | Atencion temporal sobre U-Net | dinamica temporal con foco espacial fino | afinado desde PASTIS, segunda voz temporal decorrelacionada |
| **XGBoost sobre AlphaEarth-Italia** | Tabular sobre embedding de fundacion | resumen espectral anual (64-dim) por parcela | re-entrenado sobre el embedding AlphaEarth muestreado en Italia |

**El warm-start desde PASTIS (reciclaje).** Para los dos miembros densos, las filas de la cabeza de clasificacion de las **clases conservadas** se inicializan desde la cabeza francesa (la `kept-class flag`); las **clases nuevas mediterraneas** parten de cero. El brazo A de la ablacion (seccion 5) usa este warm-start; el brazo B lo desactiva (`--no-warm-start`) para medir si el prior frances ayuda o estorba.

**La fenologia italiana.** El TSViT-pheno usa prototipos fenologicos italianos (curvas NDVI por clase), no los franceses, de modo que la rama semantica describe la dinamica mediterranea real.

**El Voting-3 a nivel parcela.** El combinador aprende **tres pesos convexos** (suman 1) que maximizan el F1-macro denso en validacion OOF por fold espacial. Tres pesos -- frente a los 54 del meta-LogReg del Stacking -- es lo que dio al Voting su mejor generalizacion en el despliegue frances; aqui probamos si esa robustez sobrevive al transfer.

In [ ]:
# Learned Voting-3 weights from the real report (interpretability of the committee).
if HAS_REPORT and report.get('voting_weights'):
    weights = report['voting_weights']
    wdf = pl.DataFrame({'miembro': list(weights.keys()),
                        'peso': [round(float(v), 4) for v in weights.values()]}
                       ).sort('peso', descending=True)
    display(wdf)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(wdf['miembro'].to_list()[::-1], wdf['peso'].to_list()[::-1], color='#6a1b9a')
    ax.set_xlabel('peso convexo (suma = 1)')
    ax.set_title('Pesos aprendidos del Voting-3 sobre Italia')
    ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show(); plt.close(fig)
    if report.get('voting_oof_f1_macro') is not None:
        print(f"F1-macro OOF (spatial-CV) del Voting-3: {report['voting_oof_f1_macro']}")
else:
    pending('pesos aprendidos del Voting-3 y su F1-macro OOF (los reporta el runner).')


## 4. Resultados por clase (fino y coarse)

El detalle por clase es donde el transfer mediterraneo cuenta su verdad. Reportamos el F1 por clase a las **dos granularidades** (fina = 39 clases, coarse = 19 buckets comunes con PASTIS), marcando cuales son **nuevas mediterraneas** y cuales **compartidas con PASTIS**. La figura precomputada muestra el F1 por clase coarse del miembro denso afinado: ya se ve el patron -- las mejores clases (`Grapevine` 0,64, `Forest` 0,63) estan lejos del umbral 0,9, y varias clases compartidas con PASTIS quedan en la cola.

In [ ]:
# Real per-class F1 figure (precomputed, coarse, run2 dense member).
show_fig(FIGS_DIR / 'fig2_f1_por_clase.png',
         'F1 por clase coarse del miembro denso afinado (azul = compartida con '
         'PASTIS, naranja = nueva mediterranea). La linea verde 0,9 es el objetivo '
         'espejo del campeon frances; ninguna clase lo alcanza -- el transfer es dificil.')


In [ ]:
# Per-class F1/IoU table (fine + coarse) from the real report, with shared/new flag.
if HAS_REPORT and report.get('voting_per_class'):
    pc = pl.DataFrame(report['voting_per_class'])
    keep = [c for c in ['leaf', 'is_new', 'f1', 'coarse_f1', 'iou', 'support'] if c in pc.columns]
    pc = pc.select(keep)
    rename = {'leaf': 'clase', 'is_new': 'es_nueva', 'f1': 'f1_fino',
              'coarse_f1': 'f1_coarse', 'iou': 'iou_fino', 'support': 'soporte_px'}
    pc = pc.rename({k: v for k, v in rename.items() if k in pc.columns})
    sort_col = 'f1_fino' if 'f1_fino' in pc.columns else pc.columns[2]
    pc = pc.sort(sort_col, descending=True)
    with pl.Config(tbl_rows=45):
        display(pc)
    if 'es_nueva' in pc.columns and 'f1_fino' in pc.columns:
        new_good = pc.filter((pl.col('es_nueva')) & (pl.col('f1_fino') >= 0.5)).height
        shared_good = pc.filter((~pl.col('es_nueva')) & (pl.col('f1_fino') >= 0.5)).height
        print(f'Clases NUEVAS mediterraneas con F1 >= 0.5: {new_good}')
        print(f'Clases COMPARTIDAS con PASTIS con F1 >= 0.5: {shared_good}')
        print('Si las nuevas ganan a las compartidas, el warm-start estorba (ver seccion 7).')
else:
    pending('tabla F1/IoU por clase (fino + coarse) del report.json. '
            'La figura coarse de arriba ya es real (run2).')


### Curva de descarte honesto y subconjunto F1 > 0,9

El objetivo de US-079 es **F1-macro > 0,9 sobre las mejores ~10 clases** (espejo del `france-10` 0,9069). Para localizar ese subconjunto sin trampa, ordenamos las clases por su F1 por clase (descendente) y reportamos el F1-macro de cada prefijo de `n` clases. Ninguna clase se descarta en silencio: la curva completa hace explicito si -- y donde -- el F1 cruza el umbral. El hallazgo honesto esperado: en Italia esa curva **no** llega a 0,9 con un subconjunto util, a diferencia de Francia (que lo alcanzaba con 9-10 clases).

In [ ]:
# Honest discard curve: F1-macro vs number of best classes kept (no cherry-picking).
if HAS_REPORT and report.get('discard_curve'):
    curve = pl.DataFrame(report['discard_curve'])
    ycol = 'macro_f1' if 'macro_f1' in curve.columns else 'f1_macro'
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(curve['n_classes'].to_list(), curve[ycol].to_list(), marker='o', color='#c62828')
    ax.axhline(f1_threshold, color='grey', linestyle='--', label=f'umbral {f1_threshold}')
    ax.axhline(france_champion_f1, color='#2e7d32', linestyle=':',
               label=f'campeon frances france-10 ({france_champion_f1})')
    ax.set_xlabel('n clases retenidas (mejores primero)'); ax.set_ylabel('F1-macro')
    ax.set_title('Curva de descarte honesto del Voting-3 sobre Italia')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show(); plt.close(fig)
    for thr in (0.8, 0.7, 0.6):
        n_cls = curve.filter(pl.col(ycol) >= thr)['n_classes'].max()
        print(f'Mayor subconjunto con F1-macro >= {thr}: '
              f'{n_cls if n_cls is not None else 0} clases.')
    best = report.get('best_subset_f1_over_0.9')
    if best:
        print(f"Mayor subconjunto con F1-macro >= {f1_threshold}: "
              f"{best.get('n_classes')} clases, F1 {best.get('macro_f1')}")
        if best.get('classes'):
            print('Clases:', ', '.join(map(str, best['classes'])))
    else:
        print(f'Ninguna ventana de la curva alcanza F1-macro >= {f1_threshold} '
              '(hallazgo honesto: el transfer mediterraneo no espeja a france-10).')
else:
    pending('curva de descarte honesto y conteo de clases >= 0.8/0.7/0.6 del report.json.')


## 5. Ablacion del warm-start (A/B)

La pregunta de investigacion mas importante de US-079: **el reciclaje de clases (warm-start desde PASTIS) ayuda o estorba en el Mediterraneo?** La tecnica funciono en Francia->Baltico; aqui la medimos con un A/B limpio:

- **Brazo A (con warm-start)**: las clases conservadas warm-startean desde la cabeza francesa (la `kept-class flag` activa) -- el comportamiento por defecto.
- **Brazo B (sin warm-start)**: `--no-warm-start`, toda la cabeza se inicializa al azar; el modelo aprende Italia sin el prior frances.

El veredicto sale del JSON que produce `scripts/run_us079_ablation_analysis.py` (patron `reports/us079_ablation_*.json`), que compara ambos brazos a nivel global y por clase, con foco en las **clases conservadas** (las que el warm-start toca). Si el brazo B (sin warm-start) iguala o supera al A en las clases conservadas, el reciclaje **estorba** -- el hallazgo clave que sospechamos.

In [ ]:
# Resolve the latest A/B ablation summary JSON (produced by the sibling runner).
ablation_path = latest_glob(ablation_glob)
ablation = load_json(ablation_path) if ablation_path else None
if ablation is None:
    pending(f'JSON de la ablacion A/B (patron {ablation_glob!r}). Lo produce '
            'scripts/run_us079_ablation_analysis.py cuando ambos brazos terminan de entrenar.')
else:
    print(f'Ablacion A/B cargada de: {Path(ablation_path).name}')
    arm_a = ablation.get('arm_a', {})
    arm_b = ablation.get('arm_b', {})
    summary = pl.DataFrame([
        {'brazo': 'A (con warm-start)', **{k: arm_a.get(k) for k in arm_a}},
        {'brazo': 'B (sin warm-start)', **{k: arm_b.get(k) for k in arm_b}},
    ])
    display(summary)


In [ ]:
# Global verdict of the A/B: does warm-start help (positive delta) or hurt (negative)?
if ablation is not None and ablation.get('delta') is not None:
    d = ablation['delta']
    if isinstance(d, dict):
        ddf = pl.DataFrame({'metrica': list(d.keys()),
                            'delta_A_menos_B': [round(float(v), 4) for v in d.values()]})
        display(ddf)
        fig, ax = plt.subplots(figsize=(7, 3.4))
        vals = list(d.values()); keys = list(d.keys())
        colors = ['#2e7d32' if float(v) >= 0 else '#c62828' for v in vals]
        ax.barh(keys[::-1], [float(v) for v in vals][::-1], color=colors[::-1])
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title('Delta A - B (positivo = warm-start AYUDA, negativo = ESTORBA)')
        ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show(); plt.close(fig)
    verdict = ablation.get('verdict') or ablation.get('conclusion')
    if verdict:
        display(Markdown(f'**Veredicto del A/B (del JSON):** {verdict}'))
else:
    pending('delta global A - B y veredicto del JSON de la ablacion.')


In [ ]:
# A/B comparison figures produced by the ablation runner (per-class + conserved delta).
show_fig(FIGS_DIR / 'fig_ab_per_class.png',
         'F1 por clase, brazo A vs brazo B: donde el warm-start cambia la decision.')
show_fig(FIGS_DIR / 'fig_ab_conserved_delta.png',
         'Delta A - B restringido a las clases CONSERVADAS (las que el warm-start toca). '
         'Un delta <= 0 aqui es la evidencia de que el reciclaje estorba en el Mediterraneo.')


**Lectura esperada del A/B (a confirmar con el JSON real).** Si la hipotesis del domain gap mediterraneo se sostiene, el brazo B (sin warm-start) **no empeora** -- y posiblemente mejora -- en las clases conservadas, porque el prior de la Francia atlantica las empuja hacia una fenologia equivocada. Eso explicaria la paradoja de la seccion 4 (las clases nuevas, que parten de cero, ganan a las compartidas, que arrastran el prior frances). El veredicto sale del JSON, no de esta prosa: el cuaderno lo imprime tal cual cuando el runner del A/B termina.

## 6. Comparacion original vs transfer learning

Dos comparaciones cierran el cuadro del transfer:

1. **Zero-shot vs fine-tune (delta del transfer).** La cota inferior es el **campeon frances zero-shot**: el checkpoint PASTIS aplicado tal cual a Italia, mapeando sus predicciones a las clases conservadas (las nuevas mediterraneas, que nunca vio, caen a fondo). El delta = (fine-tune) - (zero-shot) cuantifica cuanto aporta afinar de verdad sobre Italia.
2. **Paridad Francia vs Italia.** El campeon frances logro **F1-macro 0.9069 sobre `france-10`** (referencia medida de EPIC 6). Contrastar ese 0,9069 con el F1 que Italia alcanza sobre su mejor subconjunto cuantifica el **costo del domain gap mediterraneo** -- la distancia entre lo que el modelo lograba en casa y lo que logra al cruzar los Alpes.

In [ ]:
# Transfer delta (fine-tune - zero-shot) from the real report.
if HAS_REPORT and report.get('transfer_delta'):
    d = report['transfer_delta']
    ddf = pl.DataFrame({'metrica': list(d.keys()),
                        'delta_finetune_menos_zeroshot': [round(float(v), 4) for v in d.values()]})
    display(ddf)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    colors = ['#2e7d32' if float(v) >= 0 else '#c62828' for v in d.values()]
    ax.barh(list(d.keys())[::-1], [float(v) for v in d.values()][::-1], color=colors[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Delta del transfer (fine-tune - zero-shot)')
    ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show(); plt.close(fig)
else:
    pending('delta del transfer (fine-tune - zero-shot) del report.json '
            '(necesita la evaluacion zero-shot del campeon frances).')


In [ ]:
# Parity bar: France champion (france-10) vs Italy best subset, from real numbers.
italia_best = None
if HAS_REPORT:
    best = report.get('best_subset_f1_over_0.9')
    if best and best.get('macro_f1') is not None:
        italia_best = float(best['macro_f1'])
    elif report.get('voting_eval', {}).get('fine_f1_macro') is not None:
        italia_best = float(report['voting_eval']['fine_f1_macro'])
if italia_best is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ['Francia (france-10, medido)', 'Italia (mejor subconjunto)']
    vals = [france_champion_f1, italia_best]
    ax.bar(bars, vals, color=['#2e7d32', '#c62828'])
    ax.axhline(f1_threshold, color='grey', linestyle='--', label=f'objetivo {f1_threshold}')
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f'{v:.4f}', ha='center')
    ax.set_ylim(0, 1); ax.set_ylabel('F1-macro'); ax.legend()
    ax.set_title('Paridad Francia vs Italia (costo del domain gap mediterraneo)')
    plt.tight_layout(); plt.show(); plt.close(fig)
    print(f'Brecha Francia - Italia: {france_champion_f1 - italia_best:+.4f} F1-macro')
else:
    pending('F1-macro del mejor subconjunto italiano para la paridad vs france-10. '
            'La referencia francesa (0.9069) ya es un valor medido fijo.')


## 7. Analisis del reciclaje de clases

Esta seccion conecta el reciclaje con el hallazgo del A/B. De las clases italianas, un subconjunto se **conservo** (mapea a PASTIS y se warm-startea) y el resto son **nuevas mediterraneas** (parten de cero). La pregunta: **cuales resolvieron bien -- las recicladas o las nuevas?**

El patron observado en la figura coarse (seccion 4) es contraintuitivo: las **nuevas mediterraneas** `Grapevine` (0,64) y `Forest` (0,63) lideran, mientras varias **compartidas con PASTIS** -- `Meadow` (~0,21), `Corn` (~0,25), `Winter barley` (~0,04) -- quedan en la cola, **peor que clases que el modelo nunca habia visto**. Esa es la **paradoja del domain gap mediterraneo**: el warm-start no solo no ayuda a las clases conservadas, parece **anclarlas** a la fenologia atlantica equivocada. La celda mide el F1 medio de cada grupo desde el report real y lo contrasta con el veredicto del A/B (seccion 5) -- si el brazo B sin warm-start sube las conservadas, la paradoja queda confirmada de dos formas independientes.

In [ ]:
# Recycled (conserved) vs new (Mediterranean) groups: counts + mean F1 from the report.
if HAS_REPORT and report.get('voting_per_class'):
    pc = pl.DataFrame(report['voting_per_class'])
    f1col = 'f1' if 'f1' in pc.columns else ('f1_fino' if 'f1_fino' in pc.columns else None)
    if 'is_new' in pc.columns and f1col is not None:
        grp = (pc.group_by('is_new')
                 .agg(pl.len().alias('n_clases'),
                      pl.col(f1col).mean().round(4).alias('f1_medio'),
                      pl.col(f1col).max().round(4).alias('f1_max'))
                 .with_columns(pl.when(pl.col('is_new')).then(pl.lit('nuevas mediterraneas'))
                                 .otherwise(pl.lit('recicladas (warm-start PASTIS)')).alias('grupo'))
                 .select(['grupo', 'n_clases', 'f1_medio', 'f1_max']))
        display(grp)
        n_recycled = pc.filter(~pl.col('is_new')).height
        print(f'Clases recicladas (conservadas, warm-starteadas): {n_recycled}')
        print(f'Clases nuevas mediterraneas (desde cero): {pc.filter(pl.col("is_new")).height}')
        recycled_f1 = pc.filter(~pl.col('is_new'))[f1col].mean()
        new_f1 = pc.filter(pl.col('is_new'))[f1col].mean()
        if recycled_f1 is not None and new_f1 is not None and new_f1 > recycled_f1:
            print('PARADOJA CONFIRMADA: las clases NUEVAS superan en F1 medio a las '
                  'RECICLADAS -- el warm-start atlantico estorba en el Mediterraneo.')
    else:
        pending('columnas is_new / f1 en voting_per_class para separar recicladas vs nuevas.')
else:
    pending('per-clase con la bandera is_new del report.json para el analisis de reciclaje.')


## 8. Conclusiones honestas

**Que se logro.** Extendimos el comite ganador de EPIC 6 (Voting-3) al homologo italiano: afinamos los miembros densos desde el checkpoint PASTIS sobre 1438 patches en formato PASTIS, re-entrenamos el miembro tabular sobre AlphaEarth muestreado en Italia, y re-aprendimos los pesos del Voting con spatial-CV. Montamos ademas una **ablacion A/B limpia** del warm-start para responder, con evidencia, si el reciclaje ayuda o estorba.

**El transfer mediterraneo es dificil (hallazgo cientifico real).** A diferencia de Francia, donde el Voting-3 lograba **F1-macro 0,9069 sobre `france-10`** (referencia medida de EPIC 6), Italia **no espeja** ese resultado. La meta de F1 > 0,9 sobre las mejores ~10 clases no se alcanza: es un hallazgo honesto sobre el limite del transfer cross-domain, no un fallo a maquillar.

**Que clases funcionan.** Las que mejor resuelven son, paradojicamente, **nuevas mediterraneas** (`Grapevine` 0,64, `Forest` 0,63), no las compartidas con PASTIS. Las recicladas con fenologia mediterranea fuerte (`Meadow`, `Corn`, `Winter barley`) quedan en la cola.

**El hallazgo sobre el warm-start.** El reciclaje (`kept-class flag`) que **funciono en Francia->Baltico** parece **estorbar en Francia->Italia**. La ablacion A/B (seccion 5) lo mide: si el brazo B (sin warm-start) iguala o supera al A en las clases conservadas, el prior atlantico esta anclando esas clases a una fenologia equivocada. La paradoja se ve de dos formas independientes: la cola de clases compartidas (seccion 4/7) y el delta A - B (seccion 5).

**Limitaciones y trabajo futuro.**

- **Domain gap fenologico**: el clima mediterraneo desplaza los picos NDVI; un re-encuadre fenologico especifico para Italia (no heredado de Francia) es el siguiente paso natural.
- **Reciclaje selectivo**: en vez de warm-startear todas las clases conservadas, hacerlo solo donde la fenologia es comparable (cultivos lenosos como la vid) y dejar el resto desde cero -- una `kept-class flag` informada por la distancia de dominio.
- **Mas datos italianos** y/o un curriculum de transfer por etapas (atlantico -> templado -> mediterraneo) para suavizar el salto de dominio.

**Trazabilidad.** El run de MLflow (`us079-transfer-italia`) lleva los tags `data_version` + `code_version`; todas las cifras de este cuaderno provienen de `report.json`, del JSON de la ablacion A/B y de las figuras precomputadas -- sin numeros inventados. La unica constante fija es el `france_champion_f1 = 0.9069`, etiquetada como **referencia EPIC 6 medida**.